In [ ]:
library(tidyverse)
library(MAnorm2)

In [ ]:
Peak_file <- "./ATACseq_Chd6_profile_bins.xls"
Peak_data <- read.csv(Peak_file, sep = "\t", header = TRUE)

# data Hypoxia Scr / Normoxia Scr
norm <- normalize(Peak_data, count = 10:12, occupancy = 22:24)
norm <- normalize(norm, count = 4:6, occupancy = 16:18)

# Construct a bioCond for each group of samples.
conds <- list(N123 = bioCond(norm[10:12], norm[22:24], name = "N123"),
              H123 = bioCond(norm[4:6], norm[16:18], name = "H123"))

autosome <- !(Peak_data$chrom %in% c("chrX", "chrY"))
conds <- normBioCond(conds, common.peak.regions = autosome)

conds <- fitMeanVarCurve(conds, method = "local", occupy.only = TRUE)

# diff ana
res <- diffTest(conds[[1]], conds[[2]])

norm1 <- norm %>% select(chrom, start, end)
norm1$PeakID <- paste0("Peak_", row.names(norm1))
Data_Merge <- cbind(norm1, res)

Data_Merge$type <- "No change"
Data_Merge[Data_Merge$Mval >= 0.5 & Data_Merge$padj <= 0.05,]$type <- "Up"
Data_Merge[Data_Merge$Mval <= -0.5 & Data_Merge$padj <= 0.05,]$type <- "Down"

file_out <- "./N.Scr_H.Scr_DEA.tsv"
write.table(Data_Merge, file_out, sep = "\t", quote = FALSE, row.names = FALSE)


In [ ]:
path_root <- "/path_to/ATACseq/04_PeakDEA/02_DEAPeak"

Mval_thr <- 0.5
P_thr <- 0.05

## Scr Hypoxia VS Normoxia
file_Chd6 <- "./N.Scr_H.Scr_DEA.tsv"
Data_Chd6 <- read.table(file_Chd6, sep = "\t", header = TRUE)
Data_Chd6$type <- "No change"
Data_Chd6[Data_Chd6$Mval > Mval_thr & Data_Chd6$padj <= P_thr,]$type <- "Up"
Data_Chd6[Data_Chd6$Mval < -Mval_thr & Data_Chd6$padj <= P_thr,]$type <- "Down"

## Down
file_out <- file.path(path_root, "02_PeakAnno", "N.Scr_H.Scr_DEA_Down.tsv")
Data_Chd6Down <- Data_Chd6[Data_Chd6["type"] == "Down", ]
write.table(Data_Chd6Down, file_out, sep = "\t", row.names = FALSE, quote = FALSE)

## Up
file_out <- file.path(path_root, "02_PeakAnno", "N.Scr_H.Scr_DEA_Up.tsv")
Data_Chd6Up <- Data_Chd6[Data_Chd6["type"] == "Up", ]
write.table(Data_Chd6Up, file_out, sep = "\t", row.names = FALSE, quote = FALSE)
